# Huang Lab — Tissue-on-a-Chip MAE + Detection Training

**Before running:**
- Runtime → Change runtime type → **T4 GPU**
- Make sure `250918_Deepmind_CV_Collaboration` is added as a shortcut in your My Drive
- Have your W&B API key ready from https://wandb.ai/authorize
- Have a GitHub Personal Access Token ready (Settings → Developer settings → Personal access tokens → Tokens classic → New token, check **repo** scope)

## 1. Check GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU — go to Runtime → Change runtime type → T4 GPU')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_ROOT = '/content/drive/MyDrive/250918_Deepmind_CV_Collaboration'
assert os.path.isdir(DATA_ROOT), (
    f'Data folder not found at {DATA_ROOT}.\n'
    'Go to drive.google.com → Shared with me → right-click '
    '250918_Deepmind_CV_Collaboration → Organize → Add shortcut → My Drive'
)
print('✓ Data root found:', DATA_ROOT)
print(os.listdir(DATA_ROOT))

## 3. Clone repo (private — needs GitHub token)

Get a token at: https://github.com/settings/tokens/new  
Check the **repo** scope, generate, and paste below.

In [ ]:
from google.colab import userdata
import getpass, subprocess, sys, os

GITHUB_TOKEN = getpass.getpass("Paste your GitHub Personal Access Token: ")

REPO_DIR = "/content/Huang-Lab-Work"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # Repo already cloned — just pull latest changes
    print("Repo already exists — pulling latest changes...")
    remote = f"https://{GITHUB_TOKEN}@github.com/racyun/Huang-Lab-Work.git"
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", remote], check=True)
    result = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
    if result.returncode != 0:
        print("ERROR pulling:", result.stderr)
    else:
        print("✓ Repo updated:", result.stdout.strip())
else:
    # Fresh clone
    result = subprocess.run(
        ["git", "clone", f"https://{GITHUB_TOKEN}@github.com/racyun/Huang-Lab-Work.git",
         REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("ERROR:", result.stderr)
    else:
        print("✓ Repo cloned successfully")

%cd /content/Huang-Lab-Work


## 4. Install dependencies

In [ ]:
# Most deps (torch, wandb, transformers) are pre-installed in Colab.
# These are the project-specific extras the standard image lacks:
#   - torchmetrics[detection]: provides MeanAveragePrecision (used in eval)
#   - faster-coco-eval:        ~10x faster mAP backend (drop-in, identical numbers)
#   - accelerate:              required by HuggingFace transformers for from_pretrained
!pip install -q "torchmetrics[detection]" faster-coco-eval accelerate
print('✓ Detection eval deps installed')


In [ ]:
import wandb, getpass, os
WANDB_KEY = getpass.getpass("Paste your W&B API key (https://wandb.ai/authorize): ")

# Login and verify the key works RIGHT NOW before training starts
login_ok = wandb.login(key=WANDB_KEY, relogin=True)
os.environ["WANDB_API_KEY"] = WANDB_KEY

if not login_ok:
    raise ValueError("W&B login failed — the key you entered is invalid. "
                     "Go to https://wandb.ai/authorize and copy a fresh key.")

# Verify the key actually authenticates against the API
try:
    api = wandb.Api()
    viewer = api.viewer
    username = getattr(viewer, "name", None) or getattr(viewer, "username", None) or str(viewer)
    print(f"✓ W&B authenticated as: {username}")
    print("  You are good to go — Cell B will log to W&B.")
except Exception as e:
    os.environ.pop("WANDB_API_KEY", None)
    raise ValueError(
        f"W&B API call failed ({e}). "
        "Go to https://wandb.ai/authorize, copy a FRESH key, and re-run this cell."
    )


## 5. W&B login

In [ ]:
import os
DATA_ROOT = '/content/drive/MyDrive/250918_Deepmind_CV_Collaboration'

# Cache lives on local SSD for FAST reads (~10x faster than Drive).
# Tradeoff: cache is ephemeral — wiped on session disconnect.
# Mitigation: 'Restore from Drive backup' cell pulls a saved cache
# back from Drive at the start of each session.
CACHE_DIR = '/content/huang_lab_cache'
DRIVE_BACKUP = '/content/drive/MyDrive/huang_lab_backup'  # your own MyDrive (writable)

colab_config = f"""# Auto-generated Colab config — do not commit.

dataset:
  well_prefix: "W"
  well_count: 222
  zstack_subdir: "P00001"
  expected_z_slices: 140
  hybrid_folder_template: "hybrid_results_{{well_id}}"
  focus_filename_glob: "*focus_stacked.tif"
  cache_dir: "{CACHE_DIR}"
  resize:
    zstack:  [64, 64]
    focused: [224, 224]
    hybrid:  [224, 224]

  splits:
    - name: "900kpa"
      stiffness_kpa: 900.0
      zstack_root:  "{DATA_ROOT}/250811_Athchip_noninflam_900kPa"
      focused_root: "{DATA_ROOT}/20251027_2123__FocusStack_250811_Athchip_noninflam_900kPa"
      hybrid_root:  "{DATA_ROOT}/250811_Athchip_noninflam_900kPa_HybridResults"
      labels_root:  "{DATA_ROOT}/bbox_txt_for_training/900kPa"

    - name: "5kpa"
      stiffness_kpa: 5.0
      zstack_root:  "{DATA_ROOT}/250814_Athchip_non-inflam_5kPa"
      focused_root: "{DATA_ROOT}/20251027_2215__FocusStack_250814_Athchip_non-inflam_5kPa"
      hybrid_root:  "{DATA_ROOT}/250814_Athchip_non-inflam_5kPa_HybridResults"
      labels_root:  "{DATA_ROOT}/bbox_txt_for_training/5kPa"

training:
  output_dir: "/content/outputs"
  batch_size: 2          # bump to 16 on A100
  num_workers: 0         # bump to 4 on A100
  epochs: 50
  lr: 1.5e-4
  warmup_epochs: 5
  amp: true
  grad_clip: 1.0

detection:
  epochs: 50
  batch_size: 2
  num_workers: 0
  amp: true
  train_image_short_side: 800
  conf_threshold: 0.3

wandb:
  enabled: true
  project: "huang-lab-tissue-chip"
  entity: null
  log_freq: 5
"""

with open('config/colab.yaml', 'w') as f:
    f.write(colab_config)
print('config/colab.yaml written')
print(f'Cache (local SSD): {CACHE_DIR}')
print(f'Drive backup:      {DRIVE_BACKUP}')


## 6b. Restore previous run from Drive backup (cache + checkpoints)

If you trained before, this pulls your saved cache + checkpoints from Drive back to local disk so you can pick up where you left off — no rebuilding the cache, training resumes from the last epoch.

Safe to run on a fresh session — does nothing if no backup exists.

In [ ]:
import os, subprocess
DATA_ROOT    = '/content/drive/MyDrive/250918_Deepmind_CV_Collaboration'
DRIVE_BACKUP = '/content/drive/MyDrive/huang_lab_backup'  # your own MyDrive (writable)
CACHE_DIR    = '/content/huang_lab_cache'
OUTPUT_DIR   = '/content/outputs'

os.makedirs(DRIVE_BACKUP, exist_ok=True)
drive_cache   = f'{DRIVE_BACKUP}/cache'
drive_outputs = f'{DRIVE_BACKUP}/outputs'

def rsync(src, dst, label):
    if not os.path.isdir(src) or not os.listdir(src):
        print(f'  (no {label} backup found - skipping)')
        return
    os.makedirs(dst, exist_ok=True)
    r = subprocess.run(['rsync', '-a', f'{src}/', f'{dst}/'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  rsync {label} failed:', r.stderr[:200])
    else:
        size = subprocess.run(['du', '-sh', dst], capture_output=True, text=True).stdout.strip()
        print(f'  {label} restored ({size})')

print('Restoring from Drive backup...')
rsync(drive_cache,   CACHE_DIR,  'cache')
rsync(drive_outputs, OUTPUT_DIR, 'checkpoints')
print('Done.')


## 6. Write Colab config

## 7. Verify one batch loads

In [ ]:
!python3 scripts/train_pretrain.py \
    --config config/default.yaml \
    --local-config config/colab.yaml \
    --inspect-data

## 8. Pretrain — Multi-encoder MAE (50 epochs)

Trains three encoders jointly with stiffness conditioning.  
W&B logs: `train/loss`, `train/loss_focused`, `train/loss_hybrid`, `train/loss_volume`, per-encoder LRs.

In [ ]:
import glob, os

# Auto-detect latest pretrain checkpoint to resume from (if any)
ckpts = sorted(glob.glob('/content/outputs/multimae_epoch_*.pth'))
resume_flag = f'--resume {ckpts[-1]}' if ckpts else ''
if resume_flag:
    print(f'▶ Resuming from: {ckpts[-1]}')
else:
    print('▶ Starting fresh pretrain (no checkpoint found)')

# Use !python3 with -u (unbuffered) so all stdout/stderr streams live to the cell.
# {resume_flag} is IPython variable substitution from the Python scope above.
!python3 -u scripts/train_pretrain.py \
    --config config/default.yaml \
    --local-config config/colab.yaml \
    --train \
    --wandb \
    --wandb-run-name "full-222well-pretrain-50ep" \
    {resume_flag}


## 9. Detection fine-tuning — Deformable-DETR (50 epochs)

W&B logs: `detect/loss`, `eval/mAP`, `eval/AP50`, `eval/AP75`, `eval/mean_iou`.

In [ ]:
import glob
ckpts = sorted(glob.glob('/content/outputs/multimae_epoch_*.pth'))
print('Pretrain checkpoints found:', ckpts if ckpts else 'none')
print('Latest:', ckpts[-1] if ckpts else 'none (will use COCO weights for detection)')

In [ ]:
!python3 scripts/train_detect.py \
    --config config/default.yaml \
    --local-config config/colab.yaml \
    --wandb \
    --wandb-run-name "full-222well-detect-50ep"

## 10. Save outputs to Drive

In [ ]:
import os, subprocess
DATA_ROOT    = '/content/drive/MyDrive/250918_Deepmind_CV_Collaboration'
DRIVE_BACKUP = '/content/drive/MyDrive/huang_lab_backup'  # your own MyDrive (writable)
CACHE_DIR    = '/content/huang_lab_cache'
OUTPUT_DIR   = '/content/outputs'

os.makedirs(f'{DRIVE_BACKUP}/cache', exist_ok=True)
os.makedirs(f'{DRIVE_BACKUP}/outputs', exist_ok=True)

def rsync_to_drive(src, dst, label):
    if not os.path.isdir(src):
        print(f'  ({label}: nothing to save)')
        return
    r = subprocess.run(['rsync', '-a', f'{src}/', f'{dst}/'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f'  {label} -> Drive')
    else:
        print(f'  {label} failed:', r.stderr[:200])

print('Backing up run to Drive (cache + checkpoints)...')
rsync_to_drive(OUTPUT_DIR, f'{DRIVE_BACKUP}/outputs', 'checkpoints + logs')
rsync_to_drive(CACHE_DIR,  f'{DRIVE_BACKUP}/cache',   'cache')
print(f'\nBackup at: {DRIVE_BACKUP}')
print('Next session: run the Restore cell to pick up where you left off.')


## 11. Visualize predictions vs ground truth

Loads the latest detection checkpoint, runs inference on a few samples, and renders
side-by-side overlays: **lime green = ground truth**, **red = model predictions** (with
confidence score). Useful for sanity-checking that the model is actually detecting
the right things, not just looking at AP numbers.

In [ ]:
import glob, os
from IPython.display import Image, display

# Auto-detect latest detection checkpoint
ckpts = sorted(glob.glob('/content/outputs/detect/detector_epoch_*.pth'))
if not ckpts:
    ckpts = sorted(glob.glob('/content/outputs_quickdetect/detect/detector_epoch_*.pth'))
LATEST = ckpts[-1] if ckpts else None
print(f'Using checkpoint: {LATEST}')

if LATEST:
    VIZ_DIR = '/content/outputs/detect/visualizations'
    !python3 scripts/visualize_detections.py \
        --checkpoint {LATEST} \
        --config config/default.yaml \
        --local-config config/colab.yaml \
        --num-samples 8 \
        --score-threshold 0.3 \
        --output-dir {VIZ_DIR}

    print('\n=== Inline display ===')
    for f in sorted(os.listdir(VIZ_DIR)):
        if f.endswith('.png'):
            display(Image(f'{VIZ_DIR}/{f}'))
else:
    print('No detection checkpoint found. Train detection first.')


---
## ⚡ QUICK RUN: Detection only (no pretraining needed)

Skips MAE pretraining entirely — uses a COCO-pretrained ResNet-50 backbone built into Deformable-DETR.  
Detection only needs the **focused 2D images** (no z-stacks), so cache builds in ~20 min instead of ~5 hours.

**Expected time:** ~20 min (cache) + ~12 min/epoch × 20 epochs ≈ **~4 hours total**  
**Expected metrics after 20 epochs:** AP50 ~0.2–0.5, mAP ~0.1–0.3 — real, reportable numbers.

Run cells **A → B → C** in order. Everything above is untouched — come back later for the full pretraining run.

In [ ]:
# ── Cell A: Write quick-detect config ────────────────────────────────────────
import os
DATA_ROOT = '/content/drive/MyDrive/250918_Deepmind_CV_Collaboration'
CACHE_DIR = '/content/huang_lab_cache'   # local SSD, fast reads

quick_detect_config = f"""# Quick detection config — COCO backbone, 50 epochs.

dataset:
  well_prefix: "W"
  well_count: 222
  zstack_subdir: "P00001"
  expected_z_slices: 140
  hybrid_folder_template: "hybrid_results_{{well_id}}"
  focus_filename_glob: "*focus_stacked.tif"
  cache_dir: "{CACHE_DIR}"
  resize:
    zstack:  [64, 64]
    focused: [224, 224]
    hybrid:  [224, 224]

  splits:
    - name: "900kpa"
      stiffness_kpa: 900.0
      zstack_root:  "{DATA_ROOT}/250811_Athchip_noninflam_900kPa"
      focused_root: "{DATA_ROOT}/20251027_2123__FocusStack_250811_Athchip_noninflam_900kPa"
      hybrid_root:  "{DATA_ROOT}/250811_Athchip_noninflam_900kPa_HybridResults"
      labels_root:  "{DATA_ROOT}/bbox_txt_for_training/900kPa"

    - name: "5kpa"
      stiffness_kpa: 5.0
      zstack_root:  "{DATA_ROOT}/250814_Athchip_non-inflam_5kPa"
      focused_root: "{DATA_ROOT}/20251027_2215__FocusStack_250814_Athchip_non-inflam_5kPa"
      hybrid_root:  "{DATA_ROOT}/250814_Athchip_non-inflam_5kPa_HybridResults"
      labels_root:  "{DATA_ROOT}/bbox_txt_for_training/5kPa"

training:
  output_dir: "/content/outputs_quickdetect"
  batch_size: 2
  num_workers: 0
  epochs: 50
  lr: 1.5e-4
  warmup_epochs: 5
  amp: true
  grad_clip: 1.0

detection:
  epochs: 50
  batch_size: 2
  num_workers: 0
  amp: true
  train_image_short_side: 800
  conf_threshold: 0.3

wandb:
  enabled: true
  project: "huang-lab-tissue-chip"
  entity: null
  log_freq: 5
"""

with open('config/quick_detect.yaml', 'w') as f:
    f.write(quick_detect_config)
print('config/quick_detect.yaml written')


In [ ]:
# ── Cell B: Run detection (50 epochs, COCO backbone) ─────────────────────────
# Using !python3 so all output and errors are visible in the cell
!python3 scripts/train_detect.py \
    --config config/default.yaml \
    --local-config config/quick_detect.yaml \
    --wandb \
    --wandb-run-name "quick-detect-50ep-coco-backbone"

In [ ]:
# ── Cell C: Save quick-detect outputs + cache to Drive ────────────────────────
import os, json, subprocess
DATA_ROOT    = '/content/drive/MyDrive/250918_Deepmind_CV_Collaboration'
DRIVE_BACKUP = '/content/drive/MyDrive/huang_lab_backup'  # your own MyDrive (writable)
SAVE_DIR     = f'{DATA_ROOT}/colab_outputs_quickdetect'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_BACKUP}/cache', exist_ok=True)

subprocess.run(['rsync', '-a', '/content/outputs_quickdetect/', f'{SAVE_DIR}/'], check=False)
subprocess.run(['rsync', '-a', '/content/huang_lab_cache/', f'{DRIVE_BACKUP}/cache/'], check=False)
print(f'Outputs saved to: {SAVE_DIR}')
print(f'Cache backed up to: {DRIVE_BACKUP}/cache (persists across sessions)')

log_path = '/content/outputs_quickdetect/detect/detect_log.jsonl'
if os.path.exists(log_path):
    with open(log_path) as f:
        rows = [json.loads(l) for l in f if l.strip()]
    if rows:
        best = max(rows, key=lambda r: r.get('AP50', 0))
        last = rows[-1]
        print('\n' + '='*40)
        print(f'  FINAL RESULTS (epoch {last["epoch"]+1}/50)')
        print(f'  Loss:     {last["loss"]:.4f}')
        print(f'  AP50:     {last.get("AP50", 0):.4f}')
        print(f'  mAP:      {last.get("mAP", 0):.4f}')
        print(f'  mean IoU: {last.get("mean_iou", 0):.4f}')
        print(f'  Best AP50: {best.get("AP50", 0):.4f} (epoch {best["epoch"]+1})')
        print('='*40)
